In [4]:
import os
import zipfile
from glob import glob
from PIL import Image
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from transformers import ViTForImageClassification, ViTImageProcessor
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


zip_path = 'archive.zip'
extract_path = './brain_tumor_data'

if not os.path.exists(extract_path):
    print("Unzipping dataset...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("Unzipping complete!")

train_path = os.path.join(extract_path, 'Training')
test_path = os.path.join(extract_path, 'Testing')


class BrainTumorMRIDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.image_paths = glob(os.path.join(root_dir, '*', '*.*'))
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('L')

        if self.transform:
            image = self.transform(image)

        image = image.repeat(3, 1, 1)

        label = 0 if 'no' in img_path.lower() else 1
        return image, torch.tensor(label, dtype=torch.long)


transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])


train_dataset = BrainTumorMRIDataset(train_path, transform=transform)
test_dataset = BrainTumorMRIDataset(test_path, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)


class ViTClassifier(nn.Module):
    def __init__(self, num_classes=2):
        super(ViTClassifier, self).__init__()
        self.feature_extractor = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224-in21k')
        self.vit = ViTForImageClassification.from_pretrained('google/vit-base-patch16-224-in21k')
        self.vit.classifier = nn.Sequential(
            nn.Linear(self.vit.config.hidden_size, num_classes)
        )

    def forward(self, x):
        return self.vit(pixel_values=x).logits

model = ViTClassifier().to(device)


criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


epochs = 5
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)

        outputs = model(imgs)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss:.4f}")


model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(device), labels.to(device)

        outputs = model(imgs)
        preds = torch.argmax(outputs, dim=1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())


print("\n--- Test Metrics ---")
print("Accuracy :", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred))
print("Recall   :", recall_score(y_true, y_pred))
print("F1-Score :", f1_score(y_true, y_pred))


Unzipping dataset...
Unzipping complete!


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224-in21k and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/5 - Loss: 26.8458
Epoch 2/5 - Loss: 10.8614
Epoch 3/5 - Loss: 5.5054
Epoch 4/5 - Loss: 5.2232
Epoch 5/5 - Loss: 2.5517

--- Test Metrics ---
Accuracy : 0.977116704805492
Precision: 1.0
Recall   : 0.9668874172185431
F1-Score : 0.9831649831649831


In [6]:

torch.save(model.state_dict(), 'vit_brain_tumor.pt')
print("Model saved to vit_brain_tumor.pt")


Model saved to vit_brain_tumor.pt


In [7]:
from google.colab import files
files.download('vit_brain_tumor.pt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>